In [1]:
# 1. Check GPU allocation
!nvidia-smi

# 2. Mount Google Drive to persist data and access the repository
from google.colab import drive
drive.mount('/content/drive')

Mon Aug 17 03:09:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 1. Di chuyển vào thư mục chứa custom Ultralytics đã upload
# Hãy thay thế đường dẫn bên dưới bằng vị trí thực tế của bạn
%cd /content/drive/MyDrive/ppe_cv/Pruned-QAT-for-YOLOv8-on-waste-detection-tasks-for-edge-devices
# 2. Gỡ bản gốc để tránh xung đột
!pip uninstall -y ultralytics

# 3. Cài đặt bản custom (lệnh này sẽ cài đặt thư viện vào hệ thống Colab)
!pip install -e .

/content/drive/MyDrive/ppe_cv/Pruned-QAT-for-YOLOv8-on-waste-detection-tasks-for-edge-devices
Obtaining file:///content/drive/MyDrive/ppe_cv/Pruned-QAT-for-YOLOv8-on-waste-detection-tasks-for-edge-devices
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ultralytics (pyproject.toml) ... done
  Created wheel for ultralytics: filename=ultralytics-8.3.231-0.editable-py3-none-any.whl size=11707 sha256=c884847d9060d304f47f7fc1cd8ac99acd3185b882dd6ed964653c18e19695ce
  Stored in directory: /tmp/pip-ephem-wheel-cache-qr4ip6tb/wheels/6d/54/88/b24c9a513438bdb6003f090cbfc70bfc9b5fc534848fa36efd
Successfully built ultralytics


In [3]:
# Di chuyển về thư mục gốc của dự án
%cd /content/drive/MyDrive/ppe_cv/ppe_v2

# Kiểm tra lại xem đã đứng đúng thư mục chưa (sẽ thấy thư mục scripts, file data.yaml...)
!ls

/content/drive/MyDrive/ppe_cv/ppe_v2
assets	     demo.py			runs		 yolov8_baseline.ipynb
bench.ipynb  experiments_dagshub.ipynb	scripts		 yolov8_prune.ipynb
config.py    mlflow.db			weights
configs      README.md			yolo11n.pt
data.yaml    requirements.txt		yolov5_au.ipynb


In [4]:
!pip install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 105.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
import os

# 3. Cho phép MLflow File Store để tránh lỗi khi ghi log
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

print("Đã thiết lập xong môi trường cho Giai đoạn 2!")

Đã thiết lập xong môi trường cho Giai đoạn 2!


In [ ]:
import os

# The path to the broken file in your repo
file_path = "/content/drive/MyDrive/ppe_cv/Pruned-QAT-for-YOLOv8-on-waste-detection-tasks-for-edge-devices/ultralytics/nn/tasks_pruned.py"

# Read the current code
with open(file_path, "r", encoding="utf-8") as f:
    code = f.read()

# Replace the broken {name} variable with model.{i}
fixed_code = code.replace("f'{name}.m.", "f'model.{i}.m.")

# Write the patched code back
with open(file_path, "w", encoding="utf-8") as f:
    f.write(fixed_code)

print("✅ Successfully patched NameError in tasks_pruned.py!")

✅ Successfully patched NameError in tasks_pruned.py!


In [11]:
!python scripts/prune.py \
    --weights /content/drive/MyDrive/ppe_cv/ppe_v2/runs/train-sparsity7/weights/last.pt \
    --prune-ratio 0.20 \
    --model-size s \
    --cfg /content/drive/MyDrive/ppe_cv/Pruned-QAT-for-YOLOv8-on-waste-detection-tasks-for-edge-devices/ultralytics/cfg/models/v8/yolov8.yaml \
    --save-dir weights

Pruning gamma values should be less than 0.9263, yours is 0.4009
The corresponding pruning ratio should be less than 0.782, yours is 0.200
|	layer name               |         origin channels     |         remaining channels  |
|	model.0.bn               |         32                  |         32                  |
|	model.1.bn               |         64                  |         64                  |
|	model.2.cv1.bn           |         64                  |         64                  |
|	model.2.cv2.bn           |         64                  |         64                  |
|	model.2.m.0.cv1.bn       |         32                  |         32                  |
|	model.2.m.0.cv2.bn       |         32                  |         32                  |
|	model.3.bn               |         128                 |         126                 |
|	model.4.cv1.bn           |         128                 |         128                 |
|	model.4.cv2.bn           |         128                 |  

In [7]:
import sys
import os
import torch

# 1. Khai báo đường dẫn tới thư mục repo custom Ultralytics của bạn
# (Dựa theo log trước đó của bạn: Pruned-QAT-for-YOLOv8-on-waste-detection-tasks-for-edge-devices)
custom_repo_path = "/content/drive/MyDrive/ppe_cv/Pruned-QAT-for-YOLOv8-on-waste-detection-tasks-for-edge-devices"

if custom_repo_path not in sys.path:
    sys.path.insert(0, custom_repo_path)

In [12]:
import ultralytics

pruned_path = "/content/drive/MyDrive/ppe_cv/ppe_v2/weights/pruned_r0.20.pt"

if os.path.exists(pruned_path):
    size_mb = os.path.getsize(pruned_path) / (1024 * 1024)
    checkpoint = torch.load(pruned_path, map_location="cpu", weights_only=False)

    print(f"✅ Cắt tỉa thành công!")
    print(f"   - File lưu tại: {pruned_path} ({size_mb:.2f} MB)")
    print(f"   - Keys trong Checkpoint: {list(checkpoint.keys())}")
    print(f"   - Số lượng lớp BN đã tạo mask: {len(checkpoint.get('maskbndict', {}))}")
else:
    print(f"❌ Không tìm thấy file {pruned_path}. Vui lòng kiểm tra lại log bước Prune.")

✅ Cắt tỉa thành công!
   - File lưu tại: /content/drive/MyDrive/ppe_cv/ppe_v2/weights/pruned_r0.20.pt (28.78 MB)
   - Keys trong Checkpoint: ['model', 'maskbndict']
   - Số lượng lớp BN đã tạo mask: 57


In [13]:
# Đánh giá chỉ số mAP của mô hình pruned_r0.30.pt
!python scripts/val.py \
    --weight /content/drive/MyDrive/ppe_cv/ppe_v2/weights/pruned_r0.20.pt \
    --data data.yaml \
    --split val \
    --batch 16 \
    --imgsz 640

Average Latency per image: 8.02 ms
Approx FPS: 124.63
Ultralytics 8.3.231 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 7,464,798 parameters, 15,963 gradients, 21.5 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.1 ms, read: 60.1±18.6 MB/s, size: 80.2 KB)
val: Scanning /content/drive/MyDrive/ppe_cv/train_data/valid/labels.cache... 229 images, 19 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 229/229 2.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 2.9it/s 5.2s
                   all        229       2019      0.127     0.0279     0.0655       0.04
                     0         93        201          0          0          0          0
                     1         78        157          0          0          0          0
                     2        130        337          1    0.00593      0.503      0.328
                     3         32         65          0 

In [14]:
import mlflow
import os
import getpass

DAGSHUB_USER = input("Nhập DagsHub Username: ")
DAGSHUB_TOKEN = getpass.getpass("Nhập DagsHub Token: ")
DAGSHUB_REPO = input("Nhập tên DagsHub Repo (VD: ppe-project): ")

os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USER
os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN
os.environ["MLFLOW_TRACKING_URI"] = f"https://dagshub.com/{DAGSHUB_USER}/{DAGSHUB_REPO}.mlflow"

Nhập DagsHub Username: TQanh23
Nhập DagsHub Token: ··········
Nhập tên DagsHub Repo (VD: ppe-project): ppe_v2


In [15]:
mlflow.set_experiment("YOLOv8s_Structured_Pruning")

with mlflow.start_run(run_name="Phase2_Pruning_r0.20"):
    # Log tham số cắt tỉa
    mlflow.log_params({
        "mode": "pruning",
        "prune_ratio": 0.20,
        "model_size": "m",
        "source_weights": "/content/drive/MyDrive/ppe_cv/ppe_v2/runs/train-sparsity7/weights/last.pt"
    })

    # Upload file checkpoint đã cắt tỉa
    if os.path.exists("/content/drive/MyDrive/ppe_cv/ppe_v2/weights/pruned_r0.20.pt"):
        mlflow.log_artifact("weights/pruned_r0.30.pt", artifact_path="pruned_checkpoints")
        print("✅ Đã log checkpoint pruned_r0.30.pt lên MLflow!")

2026/08/17 03:15:49 INFO mlflow.tracking.fluent: Experiment with name 'YOLOv8s_Structured_Pruning' does not exist. Creating a new experiment.


✅ Đã log checkpoint pruned_r0.30.pt lên MLflow!
🏃 View run Phase2_Pruning_r0.20 at: https://dagshub.com/TQanh23/ppe_v2.mlflow/#/experiments/16/runs/63baf1dd23784266843c67134d27ebba
🧪 View experiment at: https://dagshub.com/TQanh23/ppe_v2.mlflow/#/experiments/16
